In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import sys 
sys.path.append("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline")
from config.constants import MIMIC_IV_PATH

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s", force=True)

%load_ext autoreload
%autoreload 2

# Check Sina's splits

In [ ]:
import zipfile, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import glob
REPO_ROOT = Path.cwd().parent

base        = REPO_ROOT / "saved_data"
folds_dir   = base / "folds"
cohorts_dir = base / "cohorts" / "DTB" /"new"
zip_path    = base / "folds_Sina" / "split_ids.zip"
mimic_pts   = pd.read_csv(Path(MIMIC_IV_PATH) / "hosp" / "patients.csv.gz")



def load_sina(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return {n: set(int(l) for l in zf.read(f"split_ids/{n}.txt").decode().splitlines() if l.strip())
                for n in ["train", "tuning", "held_out"]}
        
sina = load_sina(zip_path)

train,tuning,held_out = sina['train'],sina['tuning'],sina['held_out']
print("all mimic patients count:",mimic_pts.subject_id.nunique())
print("patients in Sina's splits:",len(train) + len(tuning) + len(held_out))
print("len training",len(train),"len validation",len(tuning),"len test",len(held_out))

In [ ]:
# check cohorts to see if Sina's cohorts are usable
files = sorted(glob.glob(str(cohorts_dir / "*.csv.gz")))
records = []
all_cohort_pats = set()
for f in files:#[:10]:
    cohort_name = Path(f).name.replace("cohort_", "").replace(".csv.gz", "")
    df = pd.read_csv(f, usecols=["subject_id", "hadm_id"])
    pats = set(df.subject_id.unique().tolist())
    all_cohort_pats |= pats

    n = len(pats)
    n_a = df.hadm_id.nunique()
    n_train = len(pats & sina["train"])
    n_val   = len(pats & sina["tuning"])
    n_test  = len(pats & sina["held_out"])
    n_uncovered = n - n_train - n_val - n_test

    records.append({
        "cohort": cohort_name,
        "n": n,
        "n_a": n_a,
        "n_train": n_train,
        "n_val": n_val,
        "n_test": n_test,
        "n_uncovered": n_uncovered,
    })

df_sina = pd.DataFrame(records).sort_values("n_test")

all_mimic = set(mimic_pts.subject_id.tolist())
not_in_cohort = all_mimic - all_cohort_pats

print(f"total cohorts        : {len(df_sina)}")
print(f"any uncovered pts    : {(df_sina.n_uncovered > 0).sum()}")
print(f"all MIMIC patients   : {len(all_mimic)}")
print(f"in some cohort       : {len(all_cohort_pats)}")
print(f"NOT in any cohort    : {len(not_in_cohort)} ({100*len(not_in_cohort)/len(all_mimic):.1f}%)")


In [ ]:
import numpy as np
import pandas as pd

BANDS  = [0, 1, 2, 3, 5, 10, np.inf]
LABELS = ["≤1%", "≤2%", "≤3%", "≤5%", "≤10%", ">10%"]

def add_deviation(d, target=(80, 10, 10)):
    d = d.assign(
        train_pct=100 * d.n_train / d.n,
        val_pct=100 * d.n_val / d.n,
        test_pct=100 * d.n_test / d.n,
    )
    d = d.assign(max_dev=np.maximum.reduce([
        (d.train_pct - target[0]).abs(),#
        (d.val_pct - target[1]).abs(),
        (d.test_pct - target[2]).abs(),
    ]))
    d = d.assign(band=pd.cut(d.max_dev, bins=BANDS, labels=LABELS, include_lowest=True))
    return d


In [ ]:
def plot_split(d, title="", target=(80, 10, 10), xlim=(60, 100), ylim=(0, 25)):
    band_counts = d.band.value_counts().reindex(LABELS, fill_value=0)

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].bar(band_counts.index, band_counts.values, color="#4c72b0")
    ax[0].set(title=f"{title}: deviation from {target[0]}/{target[1]}/{target[2]}", xlabel="max abs deviation", ylabel="cohorts")
    for i, c in enumerate(band_counts.values):
        ax[0].text(i, c + 0.5, str(c), ha="center")

    sc = ax[1].scatter(d.train_pct, d.test_pct, c=d.n, cmap="viridis", norm=LogNorm(), s=25)
    ax[1].scatter([target[0]], [target[2]], color="red", marker="*", s=300, label=f"target {target[0]}/{target[2]}")
    ax[1].set(title=f"{title}: train% vs test% (color = n)", xlabel="train %", ylabel="test %")
    ax[1].set_xlim(*xlim); ax[1].set_ylim(*ylim)
    ax[1].legend()
    fig.colorbar(sc, ax=ax[1], label="cohort size (n)")
    fig.tight_layout()
    plt.show()


In [ ]:
df_sina = add_deviation(df_sina)
plot_split(df_sina, title="Sina splits")

# Optimization

In [ ]:
#Goal: one global 80/10/10 patient split so a single pretrained model is leakage-free for all 4,648 cohorts (47,664 patients) — instead of one model per cohort.
#Objective: assign each patient to exactly one group, minimizing total per-cohort deviation from 80/10/10.
#Baseline (Sina's split): covers all patients, but small cohorts drift far from 80/10/10.
#Tried (Approach A): exact ILP with min-size constraints via Gurobi (~143k binary vars) → intractable (hours, ~50% gap) and distorts small cohorts.
#Chose (Approach B): pure minimum-error heuristic (local search) — no constraints, no solver; drop small cohorts post-hoc (keep ≥10 test patients).
#Result: obj 35.12; 4,112/4,648 cohorts within 1% of target; avg deviation <1%

In [ ]:
from flab_cohorts.utils.split_optimization import SplitOptimizer

opt = SplitOptimizer(cohorts_dir=cohorts_dir, seed=42)

opt.load_cohorts()
opt.build_index()
opt.heuristic(patience=10, tol=1e-6, max_passes=200)

all_mimic = set(mimic_pts.subject_id.tolist())
opt.save_global_split(base / "folds_global" / "split_ids", all_mimic)

df_opt = opt.summary()


In [ ]:
df_opt = add_deviation(df_opt)
plot_split(df_opt, title="Optimized")

In [ ]:
# Repeat across seeds to confirm the objective is stable (robust to initialization).
objs = []
for seed in range(5):
    o = SplitOptimizer(cohorts_dir=cohorts_dir, seed=seed)
    o.load_cohorts(); o.build_index(); o.heuristic()
    objs.append(o._ratio_obj(o.count, o.n).sum())
print("objectives across seeds:", [round(x, 3) for x in objs])

# Stratified split + Global/Local split

In [ ]:
from flab_cohorts.utils.split_optimization import generate_optimized_folds

opt = generate_optimized_folds(cohorts_dir=cohorts_dir,
                               folds_dir=folds_dir,
                               seed=42,
                               first_adm_only=True,
                               patience=10)

df_opt = opt.summary()


In [ ]:
df_opt = add_deviation(df_opt, target=(64, 16, 20))
plot_split(df_opt, title="Stratified Optimized", target=(64, 16, 20), xlim=(50, 90),ylim = (10, 30))

# Check New Splits

Global should be 80/10/10 

Each cohort should satisfy 64/16/20 ratio

The splits should be stratified

In [ ]:
import pickle
import numpy as np
import pandas as pd

seed = 42
first_adm_only = True
suffix = "_firstadm" if first_adm_only else ""

def pos_neg(hadms, label_map):
    if len(hadms) == 0:
        return 0, 0
    labels = label_map.loc[hadms[:, 1]]
    return int((labels == 1).sum()), int((labels == 0).sum())

records = []

for cohort_dir in sorted(folds_dir.iterdir()):
    cohort_name = cohort_dir.name
    fold_path = cohort_dir / f"seed_{seed}{suffix}" / "fold_0.pkl"
    cohort_file = cohorts_dir / f"{cohort_name}.csv.gz"
    if not fold_path.exists() or not cohort_file.exists():
        continue

    with open(fold_path, "rb") as f:
        train_hadms, val_hadms, test_hadms = pickle.load(f)

    cohort = pd.read_csv(cohort_file, usecols=["subject_id", "hadm_id", "admittime", "label"])
    if first_adm_only:
        cohort = cohort.sort_values("admittime").drop_duplicates("subject_id", keep="first")
    label_map = cohort.set_index("hadm_id")["label"]

    n_pos_train, n_neg_train = pos_neg(train_hadms, label_map)
    n_pos_val, n_neg_val = pos_neg(val_hadms, label_map)
    n_pos_test, n_neg_test = pos_neg(test_hadms, label_map)

    n_train, n_val, n_test = len(train_hadms), len(val_hadms), len(test_hadms)
    n_total = n_train + n_val + n_test
    n_pos_total = n_pos_train + n_pos_val + n_pos_test

    records.append({
        "cohort": cohort_name,
        "n_total": n_total,
        "n_train": n_train, "n_val": n_val, "n_test": n_test,
        "train_pct": 100 * n_train / n_total if n_total else np.nan,
        "val_pct": 100 * n_val / n_total if n_total else np.nan,
        "test_pct": 100 * n_test / n_total if n_total else np.nan,
        "n_pos_total": n_pos_total,
        "pos_ratio_total": n_pos_total / n_total if n_total else np.nan,
        "pos_ratio_train": n_pos_train / n_train if n_train else np.nan,
        "pos_ratio_val": n_pos_val / n_val if n_val else np.nan,
        "pos_ratio_test": n_pos_test / n_test if n_test else np.nan,
    })

df_all_folds = pd.DataFrame(records)
df_all_folds


In [ ]:
def load_split(split_dir):
    return {
        key: set(int(l) for l in (split_dir / f"{key}.txt").read_text().splitlines() if l.strip())
        for key in ["train", "tuning", "held_out"]
    }

In [ ]:
import pickle
import numpy as np
import pandas as pd

seed = 42
first_adm_only = True
suffix = "_firstadm" if first_adm_only else ""

records = []
pooled = {"train": set(), "val": set(), "test": set()}

for cohort_dir in sorted(folds_dir.iterdir()):
    fold_path = cohort_dir / f"seed_{seed}{suffix}" / "fold_0.pkl"
    if not fold_path.exists():
        continue
    with open(fold_path, "rb") as f:
        train_hadms, val_hadms, test_hadms = pickle.load(f)

    n_train, n_val, n_test = len(train_hadms), len(val_hadms), len(test_hadms)
    n = n_train + n_val + n_test
    if n == 0:
        continue

    records.append({
        "cohort": cohort_dir.name,
        "n": n,
        "n_train": n_train, "n_val": n_val, "n_test": n_test,
        "train_pct": 100 * n_train / n,
        "val_pct": 100 * n_val / n,
        "test_pct": 100 * n_test / n,
    })

    pooled["train"] |= set(train_hadms[:, 0].tolist())
    pooled["val"]   |= set(val_hadms[:, 0].tolist())
    pooled["test"]  |= set(test_hadms[:, 0].tolist())


tot_pooled = sum(len(v) for v in pooled.values())
print("pooled across cohort patients only (expect ~64/16/20, NOT 80/10/10):")
for k in ["train", "val", "test"]:
    print(f"  {k:6s}: {len(pooled[k]):>6} ({100*len(pooled[k])/tot_pooled:.2f}%)")

g = load_split(base / "folds_global" / "split_ids")
tot_g = sum(len(v) for v in g.values())
print("\nglobal split (includes non-cohort padding, expect 80/10/10):")
for k, kk in zip(["train", "val", "test"], ["train", "tuning", "held_out"]):
    print(f"  {k:6s}: {len(g[kk]):>6} ({100*len(g[kk])/tot_g:.2f}%)")


In [ ]:
g    = load_split(base / "folds" / "global_split_ids")     # my global split
sina = load_sina(base / "folds_Sina" / "split_ids.zip")    # Sina's split

# 1) global sizes / ratios
def ratios(d):#
    
    tot = sum(len(v) for v in d.values())
    return {k: f"{len(v)} ({100*len(v)/tot:.2f}%)" for k, v in d.items()} | {"total": tot}
print("global:", ratios(g))
print("sina  :", ratios(sina))

# 2) how many patients changed group (over the shared patients)
lbl_g    = {p: k for k, v in g.items()    for p in v}
lbl_sina = {p: k for k, v in sina.items() for p in v}
shared = set(lbl_g) & set(lbl_sina)
moved = sum(lbl_g[p] != lbl_sina[p] for p in shared)
print(f"\nshared patients: {len(shared)}, changed group: {moved} ({100*moved/len(shared):.2f}%)")


In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

cohort_name = "A08-A41"
seed, first_adm_only = 42, True
suffix = "_firstadm" if first_adm_only else ""

C = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
KEYS = ["train", "val", "test"]
GKEYS = ["train", "tuning", "held_out"]
NONE_C, OTHER_C, THIS_C = "#e1e0d9", "#c3c2b7", "#2a78d6"
MUTED, GRID, INK = "#898781", "#e1e0d9", "#0b0b0b"

with open(folds_dir / cohort_name / f"seed_{seed}{suffix}" / "fold_0.pkl", "rb") as f:
    splits = pickle.load(f)

cohort = pd.read_csv(cohorts_dir / f"{cohort_name}.csv.gz",
                     usecols=["subject_id", "hadm_id", "admittime", "label"])
if first_adm_only:
    cohort = cohort.sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first")
label_map = cohort.set_index("hadm_id")["label"]

d = pd.DataFrame([{"subject_id": sid, "hadm_id": hid, "split": k, "label": int(label_map.loc[hid])}
                  for k, s in zip(KEYS, splits) for sid, hid in s])

g = load_split(folds_dir / "global_split_ids")
gmap = {p: k for k, gk in zip(KEYS, GKEYS) for p in g[gk]}
agree = (d.split == d.subject_id.map(gmap)).mean()

d = d.sort_values(["split", "label"],
                  key=lambda c: c.map({k: i for i, k in enumerate(KEYS)}) if c.name == "split" else c)
d = d.reset_index(drop=True)

n = len(d)
ncol = int(np.ceil(np.sqrt(n * 2.2)))
d["x"], d["y"] = d.index % ncol, d.index // ncol
size = max(8, min(120, 9000 / max(n, 1)))

cohort_pats = set(d.subject_id)
all_cohort = set(opt.all_pats)
tot = np.array([len(g[k]) for k in GKEYS], dtype=float)
this = np.array([len(cohort_pats & g[k]) for k in GKEYS], dtype=float)
other = np.array([len(all_cohort & g[k]) for k in GKEYS], dtype=float) - this
none = tot - this - other

fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 2, width_ratios=[1.5, 1], hspace=0.45, wspace=0.25)
ax0, ax1, ax2, ax3 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), \
                     fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])

for k in KEYS:
    m = d[(d.split == k) & (d.label == 0)]
    ax0.scatter(m.x, -m.y, s=size, marker="s", color=C[k], alpha=0.3, linewidths=0)
    p = d[(d.split == k) & (d.label == 1)]
    ax0.scatter(p.x, -p.y, s=size, marker="s", color=C[k], edgecolors=INK, linewidths=1.8)

ax0.set(title=f"{cohort_name} — {n} patients (1 square = 1 patient)", xticks=[], yticks=[])
ax0.set_aspect("equal")
ax0.legend(handles=[Patch(facecolor=C[k], alpha=0.3, label=k) for k in KEYS] +
                   [Line2D([], [], marker="s", ls="", markerfacecolor="none",
                           markeredgecolor=INK, markeredgewidth=1.8, markersize=9, label="positive")],
           frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.02))


rate = [d[d.split == k].label.mean() for k in KEYS]
npos = [int(d[d.split == k].label.sum()) for k in KEYS]
ax1.bar(KEYS, rate, color=[C[k] for k in KEYS], width=0.55)
ax1.axhline(d.label.mean(), color=MUTED, ls="--", lw=1)
ax1.text(2.45, d.label.mean(), f" cohort {d.label.mean():.2f}", color=MUTED, va="center", fontsize=9)
for i, (r, np_) in enumerate(zip(rate, npos)):
    ax1.text(i, r, f"{r:.2f}\n({np_} pos)", ha="center", va="bottom", fontsize=9, color=INK)
ax1.set(title="Positive rate by split", ylabel="positive rate", ylim=(0, max(rate) * 1.35))

left = np.zeros(3)
for vals, c, lbl in [(none, NONE_C, "in no cohort"),
                     (other, OTHER_C, "other cohorts"),
                     (this, THIS_C, cohort_name)]:
    ax2.barh(GKEYS, vals, left=left, color=c, height=0.55, label=lbl)
    left += vals
for i, (t, x) in enumerate(zip(tot, this)):
    ax2.text(t + tot.max() * 0.01, i, f"{int(t):,} total  ·  {int(x)} from cohort",
             va="center", fontsize=9, color=MUTED)
ax2.set(title="Global split composition (all MIMIC patients)", xlabel="patients",
        xlim=(0, tot.max() * 1.42))
ax2.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18))

share = 100 * this / tot
baseline = 100 * len(cohort_pats) / tot.sum()
ax3.bar(GKEYS, share, color=THIS_C, width=0.55)
ax3.axhline(baseline, color=MUTED, ls="--", lw=1)
ax3.text(2.45, baseline, f" if evenly spread ({baseline:.3f}%)", color=MUTED, va="center", fontsize=9)
for i, s in enumerate(share):
    ax3.text(i, s, f"{s:.3f}%\n({int(this[i])})", ha="center", va="bottom", fontsize=9, color=INK)
ax3.set(title=f"{cohort_name}'s share of each global split", ylabel="% of that global split",
        ylim=(0, max(share.max(), baseline) * 1.4))

for a, axis in [(ax1, "y"), (ax2, "x"), (ax3, "y")]:
    a.grid(axis=axis, color=GRID, lw=0.8)
    a.set_axisbelow(True)
for a in (ax0, ax1, ax2, ax3):
    for s in a.spines.values():
        s.set_visible(False)
    a.tick_params(colors=MUTED, length=0)

fig.suptitle(f"agreement with global split: {agree:.0%}", y=0.02, fontsize=9, color=MUTED)
plt.show()
